# Criar parquets dos ajustes manuais

Este notebook le os arquivos de `base/csv` apenas como entrada e cria os arquivos parquet em `base/parquet`, sem modificar os CSVs originais.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

ROOT = Path.cwd()

if not (ROOT / "base" / "csv").exists():
    ROOT = Path(r"C:\Users\A1CLNDO\OneDrive - Volkswagen AG\Trabalho\Projetos\3 - Ronda\Códigos Extras\Documentação\Análisando Ajustes Manuais")

CSV_DIR = ROOT / "base" / "csv"
PARQUET_DIR = ROOT / "base" / "parquet"

CSV_DIR, PARQUET_DIR

In [ ]:
COLUMN_MAP = {
    "CHAPA": "CHAPA",
    "CODIGO_RELOGIO": "CÓDIGO_RELOGIO",
    "DESCRIÇAO_RELOGIO": "DESCRICAO_RELOGIO",
    "NOME_FUNCIONARIO": "NOME_COLABORADOR",
    "FILIAL": "FILIAL",
    "DATA_ACESSO": "DATA_ACESSO",
    "HORA_ACESSO": "HORA_ACESSO",
    "SEQUENCIA_ACESSO": "SEQUENCIA_ACESSO",
    "DIRACAO_ACESSO": "DIREÇÃO_ACESSO",
    "TIPO_ACESSO": "TIPO_ACESSO",
    "USO_MARCAÇÃO": "USU_MARCAÇÃO",
    "PLANTA": "CODIGO_PLANTA",
}

OUTPUT_COLUMNS = [
    "CHAPA",
    "CÓDIGO_RELOGIO",
    "DESCRICAO_RELOGIO",
    "NOME_COLABORADOR",
    "FILIAL",
    "DATA_ACESSO",
    "HORA_ACESSO",
    "SEQUENCIA_ACESSO",
    "DIREÇÃO_ACESSO",
    "TIPO_ACESSO",
    "USU_MARCAÇÃO",
    "CODIGO_PLANTA",
]

ARROW_SCHEMA = pa.schema([(column, pa.string()) for column in OUTPUT_COLUMNS])

In [ ]:
PERIODS = {
    "Horistas": [
        ("01-01-2024", "12-31-2024"),
        ("01-01-2025", "12-31-2025"),
        ("01-01-2026", "04-01-2026"),
    ],
    "Mensalistas": [
        ("01-01-2024", "03-31-2025"),
        ("04-01-2025", "09-30-2025"),
        ("10-01-2025", "12-31-2025"),
        ("01-01-2026", "04-01-2026"),
    ],
    # Terceiros usa o mesmo recorte de periodos da referencia Executivos.
    "Terceiros": [
        ("01-01-2024", "03-31-2025"),
        ("04-01-2025", "09-30-2025"),
        ("10-01-2025", "12-31-2025"),
        ("01-01-2026", "04-01-2026"),
    ],
}

CHUNK_SIZE = 500_000

In [ ]:
def parse_period(period):
    start, end = period
    return pd.to_datetime(start, format="%m-%d-%Y"), pd.to_datetime(end, format="%m-%d-%Y")


def output_path(category, period):
    start, end = period
    return PARQUET_DIR / category / f"{category}_{start}_{end}.parquet"


def normalize_chunk(chunk):
    chunk = chunk.rename(columns=COLUMN_MAP)
    return chunk.reindex(columns=OUTPUT_COLUMNS).astype("string")


def open_writers(category):
    writers = {}
    for period in PERIODS[category]:
        path = output_path(category, period)
        path.parent.mkdir(parents=True, exist_ok=True)
        writers[period] = pq.ParquetWriter(
            path,
            ARROW_SCHEMA,
            compression="snappy",
            use_dictionary=True,
        )
    return writers


def close_writers(writers):
    for writer in writers.values():
        writer.close()


def write_table(writer, frame):
    table = pa.Table.from_pandas(frame, schema=ARROW_SCHEMA, preserve_index=False)
    writer.write_table(table)

In [ ]:
def convert_category(category):
    source_files = sorted(CSV_DIR.glob(f"{category}_*.csv"))
    if not source_files:
        raise FileNotFoundError(f"Nenhum CSV encontrado para {category} em {CSV_DIR}")

    writers = open_writers(category)
    counts = {period: 0 for period in PERIODS[category]}
    parsed_periods = [(period, *parse_period(period)) for period in PERIODS[category]]

    try:
        for csv_file in source_files:
            print(f"Lendo {csv_file.name}...")
            reader = pd.read_csv(
                csv_file,
                encoding="latin1",
                dtype=str,
                keep_default_na=False,
                usecols=list(COLUMN_MAP.keys()),
                chunksize=CHUNK_SIZE,
            )

            for chunk in reader:
                chunk = normalize_chunk(chunk)
                access_dates = pd.to_datetime(chunk["DATA_ACESSO"], format="%m/%d/%Y", errors="coerce")

                for period, start, end in parsed_periods:
                    mask = access_dates.between(start, end, inclusive="both")
                    if not mask.any():
                        continue

                    frame = chunk.loc[mask, OUTPUT_COLUMNS]
                    write_table(writers[period], frame)
                    counts[period] += len(frame)
    finally:
        close_writers(writers)

    return counts


summary = {}
for category in PERIODS:
    print(f"\nConvertendo {category}")
    summary[category] = convert_category(category)

summary

In [ ]:
rows = []
for category, periods in summary.items():
    for period, count in periods.items():
        path = output_path(category, period)
        rows.append(
            {
                "categoria": category,
                "arquivo": path.name,
                "linhas": count,
                "tamanho_mb": round(path.stat().st_size / 1024 / 1024, 2),
            }
        )

pd.DataFrame(rows).sort_values(["categoria", "arquivo"])

## Consolidar ajustes manuais

Cria `df_ajustes_manuais.parquet` com o mesmo layout de `df_ronda.parquet`, usando a coluna `cargo` para diferenciar `H`, `M` e `T`.

In [ ]:
import duckdb

AJUSTES_PATH = PARQUET_DIR / "df_ajustes_manuais.parquet"
RONDA_ORIGINAL_PATH = Path("C:/Users/A1CLNDO/OneDrive - Volkswagen AG/Trabalho/Projetos/3 - Ronda/project/database/df_ronda.parquet")
RONDA_COM_MANUAL_PATH = PARQUET_DIR / "df_ronda.parquet"

CATEGORY_CARGO = {
    "Horistas": "H",
    "Mensalistas": "M",
    "Terceiros": "T",
}

selects = []
for category, cargo in CATEGORY_CARGO.items():
    source_glob = (PARQUET_DIR / category / "*.parquet").as_posix()
    selects.append(
        f"""
        SELECT
          CAST(CHAPA AS VARCHAR) AS cadastro,
          CAST(NOME_COLABORADOR AS VARCHAR) AS nome,
          CAST(FILIAL AS VARCHAR) AS filial,
          strptime(DATA_ACESSO, '%m/%d/%Y') AS data,
          CAST(TIME '00:00:00' + CAST(HORA_ACESSO AS INTEGER) * INTERVAL 1 MINUTE AS TIME) AS hora,
          CAST("DIREÇÃO_ACESSO" AS VARCHAR) AS dir,
          CAST(TIPO_ACESSO AS VARCHAR) AS tipo,
          CAST(SEQUENCIA_ACESSO AS VARCHAR) AS acesso,
          CAST(CODIGO_PLANTA AS VARCHAR) AS planta,
          CAST("CÓDIGO_RELOGIO" AS VARCHAR) AS coletor,
          CAST(DESCRICAO_RELOGIO AS VARCHAR) AS descricao_coletor,
          CAST("USU_MARCAÇÃO" AS VARCHAR) AS uso_acesso,
          '{cargo}' AS cargo,
          CAST(NULL AS VARCHAR) AS desligado,
          CAST(NULL AS VARCHAR) AS area,
          CAST(NULL AS VARCHAR) AS departamento,
          CAST(NULL AS VARCHAR) AS heranca,
          CAST(NULL AS VARCHAR) AS cargo_especifico
        FROM read_parquet('{source_glob}')
        WHERE DATA_ACESSO IS NOT NULL AND HORA_ACESSO IS NOT NULL
        """
    )

manual_sql = "\nUNION ALL\n".join(selects)

with duckdb.connect(":memory:") as con:
    con.execute("PRAGMA threads=4")
    con.execute(
        f"""
        COPY (
        {manual_sql}
        ) TO '{AJUSTES_PATH.as_posix()}' (FORMAT PARQUET, COMPRESSION SNAPPY)
        """
    )

AJUSTES_PATH

## Unir com a base Ronda

Cria o `df_ronda.parquet` consolidado na pasta dos ajustes manuais, adicionando `Manual = Não` para a base original e `Manual = Sim` para os ajustes.

In [ ]:
base_fields = [
    ("cadastro", pa.string()),
    ("nome", pa.string()),
    ("filial", pa.string()),
    ("data", pa.timestamp("ns")),
    ("hora", pa.time64("us")),
    ("dir", pa.string()),
    ("tipo", pa.string()),
    ("acesso", pa.string()),
    ("planta", pa.string()),
    ("coletor", pa.string()),
    ("descricao_coletor", pa.string()),
    ("uso_acesso", pa.string()),
    ("cargo", pa.string()),
    ("desligado", pa.string()),
    ("area", pa.string()),
    ("departamento", pa.string()),
    ("heranca", pa.string()),
    ("cargo_especifico", pa.string()),
]

base_cols = [name for name, _ in base_fields]
base_schema = pa.schema(base_fields)
full_schema = pa.schema(base_fields + [("Manual", pa.string())])


def append_parquet_with_manual(writer, source_path, manual_value, batch_size=500_000):
    pf = pq.ParquetFile(source_path)
    total = 0
    for batch in pf.iter_batches(batch_size=batch_size, columns=base_cols):
        table = pa.Table.from_batches([batch]).cast(base_schema, safe=False)
        table = table.append_column("Manual", pa.array([manual_value] * table.num_rows, type=pa.string()))
        writer.write_table(table)
        total += table.num_rows
    return total


writer = pq.ParquetWriter(RONDA_COM_MANUAL_PATH, full_schema, compression="snappy", use_dictionary=True)
try:
    linhas_original = append_parquet_with_manual(writer, RONDA_ORIGINAL_PATH, "Não")
    linhas_manual = append_parquet_with_manual(writer, AJUSTES_PATH, "Sim")
finally:
    writer.close()

linhas_original, linhas_manual, RONDA_COM_MANUAL_PATH

In [ ]:
validacao = []
for path in [AJUSTES_PATH, RONDA_COM_MANUAL_PATH]:
    pf = pq.ParquetFile(path)
    validacao.append(
        {
            "arquivo": path.name,
            "linhas": pf.metadata.num_rows,
            "colunas": pf.metadata.num_columns,
            "tamanho_mb": round(path.stat().st_size / 1024 / 1024, 2),
        }
    )

pd.DataFrame(validacao)